# QuickTalk: HellaSwag for earlier models (run 7, run 7c) + SmolLM2 reference

Scores the model zips already in your Drive (`quicktalk_run7_model.zip`, `quicktalk_run7c_model.zip`) on HellaSwag
validation (10,042 items, 4 endings each, random = 25%), plus SmolLM2-135M with the same scorer.
Runs in its OWN runtime and folder: it does not touch a run 8 training session. A T4 GPU is enough
(~10-15 min per model). If Colab will not give a second GPU while run 8 trains, run this later.

Note: these zips contain the CHAT models (sft_final.pt); the run 7 base model was deleted, so the run 7 numbers are
for the chat model. Published base-model reference (acc_norm, approximate): GPT-2 small ~30%, SmolLM2-135M ~42%.

In [ ]:
# 1. Drive, code, libraries (own folder /content/qt_eval)
import os, subprocess
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
BASE = '/content/drive/MyDrive'; OUT = f'{BASE}/hellaswag_results'; SRC = '/content/qt_eval'
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(SRC):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'claude/intelligent-ride-oxqjfy', '--filter=blob:none',
                    '--sparse', 'https://github.com/sraivante/quicktalk.git', SRC], check=True)
    subprocess.run(['git', '-C', SRC, 'sparse-checkout', 'set', 'train/colab'], check=True)
HS = f'{SRC}/train/colab/hellaswag_eval.py'
!pip -q install datasets tokenizers transformers

In [ ]:
# 2. Unzip the earlier models (local copies; the Drive zips are only read)
import zipfile, os
MODELS = {}
for run in ['quicktalk_run7', 'quicktalk_run7c']:
    z = f'{BASE}/{run}_model.zip'
    if not os.path.exists(z): print('not found, skipping:', z); continue
    d = f'/content/models/{run}'
    if not os.path.exists(f'{d}/sft_final.pt'):
        os.makedirs(d, exist_ok=True); zipfile.ZipFile(z).extractall(d)
    MODELS[run] = d; print(run, sorted(os.listdir(d)))

In [ ]:
# 3. Score our models
for run, d in MODELS.items():
    out = f'{OUT}/{run}_chat.json'
    if not os.path.exists(out):
        !python {HS} --workdir {d} --ckpt {d}/sft_final.pt --out {out}

In [ ]:
# 4. SmolLM2-135M (base and instruct) with the same scorer, for a like-for-like comparison
for m, name in [('HuggingFaceTB/SmolLM2-135M', 'smollm2_135m_base'), ('HuggingFaceTB/SmolLM2-135M-Instruct', 'smollm2_135m_instruct')]:
    out = f'{OUT}/{name}.json'
    if not os.path.exists(out):
        !python {HS} --hf-model {m} --out {out}

In [ ]:
# 5. Summary (results are kept in MyDrive/hellaswag_results)
import json, glob
for p in sorted(glob.glob(f'{OUT}/*.json')):
    r = json.load(open(p)); print(f"{p.split('/')[-1]:34} acc_norm {r['acc_norm']:.1%}  acc {r['acc']:.1%}  ({r['items']} items)")